In [1]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2025 Hammerheads Engineers sp. z o.o.
# Author: Aleksander Stanik
import sys
import os
import time
import yaml
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, '..'))

if repo_root not in sys.path:
    sys.path.append(repo_root)


import spx_python
spx_python.set_global_transparent(False)
# Initialize HTTP-based SPX client wrapper pointing to local SPX server
product_key = os.environ['SPX_PRODUCT_KEY']
wrapper = spx_python.init(address='http://localhost:8000',
                                product_key=product_key)

In [59]:
wrapper.reload_modules()

{'result': None}

In [3]:
# Create a new model for the PT100 sensor
pt_100_yaml = '''
attributes:
  temperature: 0.0
actions:
  - { saw: $attr(temperature), stop_value: 14, period: 5 }
  - { ramp: $attr(temperature), stop_value: 150, duration: 5, type: overshoot, overshoot: 5 }
  - { noise: $ext(temperature), std: 0.01, mode: proportional }
  - { contact_fault: $ext(temperature), spike_value: 500.0 }
'''

# Parse YAML and build the model
data = yaml.safe_load(pt_100_yaml)
wrapper["models"]["pt_100_1"] = data
wrapper["instances"].clear()
wrapper["instances"]["test_pt_100"] = "pt_100_1"

instance = wrapper["instances"]["test_pt_100"]

print("Available models:", wrapper["models"].keys())
print("Available instances:", wrapper["instances"].keys())

Available models: ['TemperatureSensor', 'PowerSupply', 'PIDController', 'pt_100', 'pt_100_1']
Available instances: ['test_pt_100']


In [9]:
print(instance["actions"].keys())

['saw', 'ramp', 'noise', 'contact_fault']


In [ ]:
temperatures = []
temperatures_raw = []
times = np.linspace(0, 10, 1000)

instance["polling"].disable()
# instance["actions"]["saw"].disable()

instance.reset()
instance.prepare()

temperature = instance["attributes"]["temperature"]
timer = instance["timer"]
# Run the instance for each time step
for t in times:
    timer.time = t
    instance.run()
    temperatures.append(temperature.external_value)
    temperatures_raw.append(temperature.internal_value)
    # print (f"Time: {t:.2f}s, Temperature: {temperatures[-1]:.2f}°C, Raw: {temperatures_raw[-1]:.2f}°C")

# Plotting the results
fig = go.Figure()
fig.add_trace(go.Scatter(x=times, y=temperatures, mode='lines', name='Temperature'))
fig.add_trace(go.Scatter(x=times, y=temperatures_raw, mode='lines', name='Temperature Internal'))
fig.update_layout(
    title='Change of Temperature Over Time',
    xaxis_title='Time (s)',
    yaxis_title='Temperature (°C)',
    showlegend=True
)
